# 04 · Model Selection: Logistic Regression, step by step

Notebook 03 compared 9 models and chose **Logistic Regression**. Here we build it step by step and look inside it.

**Goal:** learn from past customers what leavers have in common, then give every customer in the **test set** a
**churn probability**, and rank them so we know **who to retain first**.

Steps:
1. Load the clean data
2. Split into **train** (to learn) and **test** (to check), *before* any learning from the data
3. Prepare the features (encode text, impute, scale), learning only from **train**
4. Train **Logistic Regression** (main model)
5. Check how good it is on the test set
6. See **what drives churn** (the model's weights)
7. Build the **priority list** of customers to retain

> Run `01_data_types.ipynb` first. It creates `telco_clean.pkl`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (confusion_matrix, accuracy_score, recall_score,
                             precision_score, roc_auc_score)

df = pd.read_pickle("telco_clean.pkl")
print("Customers:", len(df))

## 1 · Choose the inputs (X) and the answer (y)

- **y** = `Churn` (1 = left, 0 = stayed). This is what we want to predict.
- **X** = everything else, except:
  - `customerID`: just a name tag, it tells nothing about behaviour
  - `TotalCharges`: it is almost exactly `tenure x MonthlyCharges` (we checked in notebook 01), so it adds no new
    information and would confuse the model's weights

In [ ]:
y = df["Churn"]
X = df.drop(columns=["Churn", "customerID", "TotalCharges"])

print("Input columns:", list(X.columns))

## 2 · Split into train and test

- **Train (80%)**: the model learns from these customers
- **Test (20%)**: the model has *never seen* these, so they act like "future" customers
- `stratify=y` keeps the same 26.5% churn rate in both parts
- `random_state=42` makes the split the same every time we run it

We split **first**, so nothing from the test customers leaks into the learning.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)

print("Train customers:", len(X_train), "| churn rate:", round(y_train.mean() * 100, 1), "%")
print("Test customers: ", len(X_test), "| churn rate:", round(y_test.mean() * 100, 1), "%")

## 3 · Prepare the features

### 3a. Turn text into numbers (one-hot encoding)

Models only understand numbers. `pd.get_dummies` turns a text column into Yes/No (1/0) columns.
Example: `Contract` becomes `Contract_One year` and `Contract_Two year`.
If both are 0, the customer is month-to-month (the dropped "baseline" group, because of `drop_first=True`).

This learns nothing from the data (the list of categories was fixed in notebook 01), so it is safe to apply to both parts.

In [ ]:
X_train = pd.get_dummies(X_train, drop_first=True, dtype=int)
X_test = pd.get_dummies(X_test, drop_first=True, dtype=int)

print("Number of input columns after encoding:", X_train.shape[1])
print("Same columns in train and test:", list(X_train.columns) == list(X_test.columns))
X_train.head()

### 3b. Fill missing numbers (median imputation)

Our data has no missing values left, but a real new customer record might.
`SimpleImputer` learns the **median from train only**, then fills gaps in both train and test.

In [ ]:
num_cols = ["tenure", "MonthlyCharges"]

imputer = SimpleImputer(strategy="median")
imputer.fit(X_train[num_cols])                      # learn medians from TRAIN only
X_train[num_cols] = imputer.transform(X_train[num_cols])
X_test[num_cols] = imputer.transform(X_test[num_cols])

print("Medians learned from train:", dict(zip(num_cols, imputer.statistics_)))

### 3c. Put numbers on the same scale

`tenure` goes 0 to 72 and `MonthlyCharges` goes about 18 to 119, while the encoded columns are only 0/1.
`StandardScaler` rescales the two number columns so the model treats them fairly.
Again: it learns the averages from **train only**.

In [ ]:
scaler = StandardScaler()
scaler.fit(X_train[num_cols])                       # learn from TRAIN only
X_train[num_cols] = scaler.transform(X_train[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])

X_train[num_cols].describe().round(2)

## 4 · Train the Logistic Regression model

**How it works (simple version):** the model gives every input a **weight**.
A positive weight pushes a customer **towards leaving**, a negative weight pushes **towards staying**.
It adds up the weights for each customer and turns the total into a **probability between 0% and 100%**.

`class_weight="balanced"`: only 26.5% of customers left, so we tell the model to pay extra attention to leavers.
Otherwise it gets lazy and predicts "stays" for almost everyone.

In [ ]:
model = LogisticRegression(class_weight="balanced", max_iter=1000)
model.fit(X_train, y_train)

print("Model trained.")

## 5 · How good is the model? (on the unseen test customers)

- `predict_proba` gives each test customer a **churn probability**
- If the probability is above 50%, we say "will leave"

In [ ]:
test_prob = model.predict_proba(X_test)[:, 1]      # probability of leaving
test_pred = (test_prob >= 0.5).astype(int)         # 1 = we predict they will leave

print(f"Recall:    {recall_score(y_test, test_pred) * 100:.1f}%  <- of the customers who really left, how many we caught")
print(f"Precision: {precision_score(y_test, test_pred) * 100:.1f}%  <- of the customers we flagged, how many really left")
print(f"Accuracy:  {accuracy_score(y_test, test_pred) * 100:.1f}%")
print(f"ROC-AUC:   {roc_auc_score(y_test, test_prob):.3f}  <- 0.5 = guessing, 1.0 = perfect ranking")

print(f"\nFor comparison, guessing 'nobody leaves' gives {(1 - y_test.mean()) * 100:.1f}% accuracy but 0% recall.")

### Confusion matrix

A simple table of right and wrong answers.

In [ ]:
cm = confusion_matrix(y_test, test_pred)
cm_table = pd.DataFrame(cm,
                        index=["Really stayed", "Really left"],
                        columns=["Predicted stay", "Predicted leave"])
cm_table

In [ ]:
plt.imshow(cm, cmap="Blues")
for i in range(2):
    for j in range(2):
        plt.text(j, i, cm[i, j], ha="center", va="center", fontsize=14)
plt.xticks([0, 1], ["Predicted stay", "Predicted leave"])
plt.yticks([0, 1], ["Really stayed", "Really left"])
plt.title("Confusion matrix (test customers)")
plt.show()

### Train vs test: is the model overfitting?

**Overfitting** = the model memorises the training customers instead of learning general patterns.
It then scores very well on train but badly on test.

If the train and test scores are **close**, the model has learned real patterns that also work on new customers.

In [ ]:
def scores(true, prob):
    # all our metrics for one set of customers
    pred = (prob >= 0.5).astype(int)
    return {
        "accuracy_%": round(accuracy_score(true, pred) * 100, 1),
        "recall_%": round(recall_score(true, pred) * 100, 1),
        "precision_%": round(precision_score(true, pred) * 100, 1),
        "roc_auc": round(roc_auc_score(true, prob), 3),
    }

train_prob = model.predict_proba(X_train)[:, 1]

train_vs_test = pd.DataFrame({
    "train": scores(y_train, train_prob),
    "test": scores(y_test, test_prob),
})
train_vs_test["gap"] = (train_vs_test["train"] - train_vs_test["test"]).round(3)
train_vs_test

**How to read it:**
- Bottom-right: leavers we **caught** (good)
- Bottom-left: leavers we **missed** (the costly mistake: they leave and we did nothing)
- Top-right: stayers we flagged by mistake (small cost: we give an offer to someone who would have stayed)

That is why **recall** is our headline number.

## 6 · What drives churn? (the model's weights)

We turn each weight into an **odds ratio** with `np.exp(weight)`:
- **above 1** = raises the chance of leaving (e.g. 2.0 = doubles the odds)
- **below 1** = lowers it (e.g. 0.3 = cuts the odds to about a third)

For Yes/No columns, it compares against the baseline group (e.g. a two-year contract vs month-to-month).
For `tenure` and `MonthlyCharges`, it is the effect of one "standard step" up (about 25 months of tenure, or about $30/month).

In [ ]:
weights = pd.DataFrame({
    "feature": X_train.columns,
    "weight": model.coef_[0],
})
weights["odds_ratio"] = np.exp(weights["weight"]).round(2)
weights = weights.sort_values("weight", ascending=False)

print("Pushes customers to LEAVE:")
print(weights.head(8).to_string(index=False))
print("\nKeeps customers (pushes to STAY):")
print(weights.tail(8).to_string(index=False))

In [ ]:
top = pd.concat([weights.head(8), weights.tail(8)])
colors = ["red" if w > 0 else "green" for w in top["weight"]]

plt.figure(figsize=(8, 6))
plt.barh(top["feature"], top["weight"], color=colors)
plt.axvline(0, color="black")
plt.gca().invert_yaxis()
plt.title("What pushes customers to leave (red) or stay (green)")
plt.xlabel("Model weight")
plt.show()

## 7 · Priority list: who should we retain first?

Not every leaver is worth the same. A customer paying $100/month is worth more than one paying $20/month.

**Priority score = churn probability x monthly charge**
= the money we *expect* to lose from this customer each month.

We use the **original** (unscaled) monthly charge here, so the score is in real dollars.

In [ ]:
priority = df.loc[X_test.index, ["customerID", "Contract", "tenure", "InternetService",
                                 "TechSupport", "OnlineSecurity", "PaymentMethod",
                                 "MonthlyCharges", "Churn"]].copy()
priority["churn_prob_%"] = (test_prob * 100).round(1)
priority["expected_loss_$"] = (test_prob * priority["MonthlyCharges"]).round(2)

# simple risk labels
priority["risk"] = pd.cut(priority["churn_prob_%"], bins=[-1, 40, 70, 101],
                          labels=["Low", "Medium", "High"])

priority = priority.sort_values("expected_loss_$", ascending=False)
priority.head(15)

In [ ]:
priority["risk"].value_counts()

### Does the list work?

If we only contact the **top 20%** of the list, how many of the real leavers do we reach?
(Random picking would reach only about 20% of them.)

In [ ]:
top_n = int(len(priority) * 0.20)
top_group = priority.head(top_n)

caught = top_group["Churn"].sum()
all_leavers = priority["Churn"].sum()
money_caught = top_group.loc[top_group["Churn"] == 1, "MonthlyCharges"].sum()
money_all = priority.loc[priority["Churn"] == 1, "MonthlyCharges"].sum()

print(f"Contacting the top {top_n} customers (20% of the test set):")
print(f"  reaches {caught} of {all_leavers} real leavers ({caught / all_leavers * 100:.1f}%)")
print(f"  covers {money_caught / money_all * 100:.1f}% of the monthly revenue that actually left")

## 8 · Save for the next notebooks

We save this first version of the model and the priority list. (The final model used by the simulation and the app is
saved in notebook 05.)

In [ ]:
import joblib

joblib.dump({"model": model, "imputer": imputer, "scaler": scaler,
             "num_cols": num_cols, "columns": list(X_train.columns)}, "churn_model.joblib")
priority.to_csv("priority_list.csv", index=False)
print("Saved churn_model.joblib and priority_list.csv")

## Key results

### How good is the model? (1,409 unseen test customers)

| Metric | Train | **Test** |
|---|---|---|
| Accuracy | 75.3% | **74.0%** |
| **Recall** (leavers caught) | 79.9% | **77.3%** (289 of 374) |
| Precision | 52.2% | **50.6%** |
| **ROC-AUC** | 0.848 | **0.839** |

- **No overfitting:** train and test scores are within about 1–3 points, so the model learned general patterns, not the training customers.

- Accuracy alone is misleading: "nobody leaves" scores 73.5% but catches nobody.

### What drives churn (odds ratios)

| Pushes to **leave** | | Keeps customers (**stay**) | |
|---|---|---|---|
| Fiber optic | **3.5x** | Two-year contract | **0.24x** (odds cut by ~76%) |
| Streaming movies / TV | 1.5x each | Longer tenure | 0.47x per ~25 months |
| Electronic check | 1.5x | One-year contract | 0.49x |
| Multiple lines | 1.4x | OnlineSecurity | 0.73x |
| Paperless billing | 1.4x | TechSupport | 0.78x |

**If a judge asks "why does MonthlyCharges keep customers when the EDA said high prices churn more?":**
a customer's price is mostly the *sum of their services*. The model already counts fiber, streaming and extra lines
(which is where the high price comes from), so `MonthlyCharges` only holds what is left over. The price effect shows up in
the fiber and streaming weights instead. Same story, split across columns.

### Priority list
- Top of the list: **month-to-month, fiber, under 1 year, no TechSupport or OnlineSecurity, paying by electronic check, around $100/month**. This matches the EDA "typical churner" exactly.
- Contacting only the **top 20%** of the list reaches **44.7% of real leavers** (random picking: about 20%) and covers **54.2% of the revenue that actually left**.

### Note on the probabilities
Because of `class_weight="balanced"`, the probabilities are pushed upwards. They are great for **ranking** customers, but
"90%" should not be read as an exact chance. The High / Medium / Low labels are for ordering, not exact forecasts.

**Next:** notebook 05 adds engineered features, tunes the hyperparameters and picks the best threshold.
Notebook 06 then uses that final model for the pricing simulation, retention playbook and $ impact.
